# RNA-seq Analysis — Module I Final Project

**Course:** Bioinformatics Methods for Transcriptomics  
**Institution:** Johns Hopkins University

## 01. Project Overview

- This project investigates changes in gene expression during mouse hippocampus development using RNA-seq data.
- The dataset contains six biological samples: three from 1-month-old mice and three from 4-month-old mice.
- The analysis uses a 3.4 Mb genomic region from mouse chromosome 7 (GRCm39) and the corresponding GENCODE vM30 annotation.

### Analysis Workflow

1. Prepare and validate the input files.
2. Build the STAR genome index.
3. Align RNA-seq reads using STAR.
4. Examine alignment statistics using STAR and SAMtools.
5. Assemble transcripts using PsiCLASS.
6. Annotate transcripts using GENCODE.
7. Analyze differential expression using Cuffdiff2.
8. Generate gene-level counts and analyze them using DESeq2.
9. Summarize and interpret the results.

The workflow includes input validation, quality checks, and documentation of the analysis parameters to support reproducibility.

## 02. Environment Setup & Dependencies

- This section prepares the computational environment and checks the software required for the RNA-seq analysis.
- Python is used for file management, data processing, and execution of external commands. The main bioinformatics analyses are performed using STAR, SAMtools, PsiCLASS, Cuffdiff2, and R/Bioconductor (DESeq2).
- Before starting the analysis, the availability of the required Python packages and bioinformatics tools is verified.
- PsiCLASS is installed in a dedicated Conda environment, while DESeq2 and the required Bioconductor packages are available in the `transcriptomics` environment.

In [47]:
# Import required Python libraries
from pathlib import Path             # Manage file and directory paths
from collections import defaultdict, Counter
import importlib.util                # Check Python package availability
import os                            # Access environment variables
import re                            # Process text patterns
import shutil                        # Locate executables
import subprocess                    # Run external commands
import sys                           # Access Python information

# Check the current Python environment
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Conda:", os.environ.get("CONDA_DEFAULT_ENV", "Not detected"))

Python: 3.11.6
Executable: /home/ianca/miniconda3/envs/transcriptomics/bin/python
Conda: transcriptomics


In [2]:
# Check availability of bioinformatics tools
TOOLS = {
    "STAR": "STAR",
    "SAMtools": "samtools",
    "Cuffdiff2": "cuffdiff",
    "junc": "junc",
}

for name, command in TOOLS.items():
    executable = shutil.which(command)
    if executable:
        print(f"[OK] {name}: {executable}")
    else:
        print(f"[NOT FOUND] {name}")

# Check tools installed in dedicated Conda environments
HOME = Path.home()

PSICLASS_EXE = (HOME / "miniconda3/envs/psiclass/bin/psiclass")
RSCRIPT_EXE = (HOME / "miniconda3/envs/transcriptomics/bin/Rscript")

for name, executable in {
    "PsiCLASS": PSICLASS_EXE,
    "Rscript": RSCRIPT_EXE,
}.items():

    if executable.is_file():
        print(f"[OK] {name}: {executable}")
    else:
        print(f"[NOT FOUND] {name}: {executable}")

[OK] STAR: /usr/bin/STAR
[OK] SAMtools: /usr/bin/samtools
[OK] Cuffdiff2: /usr/bin/cuffdiff
[NOT FOUND] junc
[OK] PsiCLASS: /home/ianca/miniconda3/envs/psiclass/bin/psiclass
[OK] Rscript: /home/ianca/miniconda3/envs/transcriptomics/bin/Rscript


In [3]:
# Check required Python packages
PYTHON_PACKAGES = ["numpy", "pandas", "matplotlib", "IPython",]

for package in PYTHON_PACKAGES:
    if importlib.util.find_spec(package) is not None:
        print(f"[OK] {package}")
    else:
        print(f"[NOT FOUND] {package}")

import numpy as np
import pandas as pd
import matplotlib as plt

[OK] numpy
[OK] pandas
[OK] matplotlib
[OK] IPython


In [4]:
# Check required R/Bioconductor packages
r_code = r'''
packages <- c("DESeq2", "GenomicFeatures", "GenomicAlignments", "Rsamtools", "SummarizedExperiment", "txdbmaker")

for (pkg in packages) {
    installed <- requireNamespace(pkg, quietly = TRUE)
    status <- if (installed) "[OK]" else "[NOT FOUND]"
    cat(status, pkg, "\n")
}
'''

if RSCRIPT_EXE.is_file():
    result = subprocess.run(
        [str(RSCRIPT_EXE), "--vanilla", "-e", r_code],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=True)

    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print("[SKIPPED] Rscript executable not found.")

[OK] DESeq2 
[OK] GenomicFeatures 
[OK] GenomicAlignments 
[OK] Rsamtools 
[OK] SummarizedExperiment 
[OK] txdbmaker 



## 03. Project Configuration

- This section defines the project directories, reference files, sample identifiers, and analysis parameters used throughout the RNA-seq workflow.
- The dataset includes six mouse hippocampus samples divided into two experimental groups: three biological replicates from 1-month-old mice and three from 4-month-old mice.
- Project paths and analysis parameters are defined centrally to ensure consistency across the different processing steps.

In [43]:
# Define project directories
REPO_DIR = Path("/mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project")
DATA_DIR = Path("/mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project")

RAW_DIR = REPO_DIR / "data_raw"
RESULTS_DIR = DATA_DIR / "results"

# Define analysis directories
STAR_DIR = RESULTS_DIR / "STAR"
STAR_INDEX_DIR = STAR_DIR / "star_index"
STAR_ALIGNMENT_DIR = STAR_DIR / "alignments"

PSICLASS_DIR = RESULTS_DIR / "PsiCLASS"
CUFFDIFF_DIR = RESULTS_DIR / "Cuffdiff"
SORTED_DIR = RESULTS_DIR / "Sorted"
DESEQ2_DIR = RESULTS_DIR / "DESeq2"
DESEQ2_RESULTS_DIR = DESEQ2_DIR / "Results"

# Define reference files
REFERENCE_FASTA = RAW_DIR / "region.fa"
GENCODE_GTF = RAW_DIR / "gencode.vM30.region.gtf"
METADATA_FILE = RAW_DIR / "metadata.tsv"

print("Repository:", REPO_DIR)
print("Data directory:", DATA_DIR)
print("Reference FASTA:", REFERENCE_FASTA)
print("GENCODE annotation:", GENCODE_GTF)
print("STAR index:", STAR_INDEX_DIR)
print("STAR alignments:", STAR_ALIGNMENT_DIR)

Repository: /mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project
Data directory: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project
Reference FASTA: /mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project/data_raw/region.fa
GENCODE annotation: /mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project/data_raw/gencode.vM30.region.gtf
STAR index: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/star_index
STAR alignments: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/alignments


In [42]:
# Define experimental groups and sample identifiers
SAMPLES_1MO = ["1mo_Rep1", "1mo_Rep2", "1mo_Rep3"]
SAMPLES_4MO = ["4mo_Rep1", "4mo_Rep2", "4mo_Rep3"]
SAMPLES = SAMPLES_1MO + SAMPLES_4MO

# Define analysis parameters
THREADS = 2
GENOME_SA_INDEX_NBASES = 9
GENOME_SA_SPARSE_D = 1

# Create sample metadata
sample_metadata = pd.DataFrame({
    "sampleID": SAMPLES,
    "condition": (
        ["1mo"] * len(SAMPLES_1MO)
        + ["4mo"] * len(SAMPLES_4MO)
    ),
})

display(sample_metadata)
print("Number of samples:", len(SAMPLES))

,sampleID,condition
0,1mo_Rep1,1mo
1,1mo_Rep2,1mo
2,1mo_Rep3,1mo
3,4mo_Rep1,4mo
4,4mo_Rep2,4mo
5,4mo_Rep3,4mo


Number of samples: 6


## 04. Input Validation & GENCODE Annotation

- Before starting the RNA-seq analysis, the input files are checked to confirm their availability and integrity.
- The dataset includes six paired-end RNA-seq samples, a reference FASTA file containing a selected region of mouse chromosome 7, and a corresponding GENCODE GTF annotation.
- The reference annotation is examined to determine the number of annotated genes. Gene identifiers are extracted from the `gene_id` attribute, and unique identifiers are counted to avoid counting multiple transcripts or exons from the same gene.
- The counting method provided in the course is also reproduced for comparison.

In [48]:
# Check reference files and sample metadata
for path in [REFERENCE_FASTA, GENCODE_GTF, METADATA_FILE]:
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f"[OK] {path.name}")

# Check sequencing files
FASTQ_FILES = {}

for sample in SAMPLES:
    fastq = RAW_DIR / f"{sample}.fastq.gz"

    if not fastq.is_file():
        raise FileNotFoundError(fastq)

    FASTQ_FILES[sample] = fastq
    print(f"[OK] {sample}")
print("\nAll input files found.")

[OK] region.fa
[OK] gencode.vM30.region.gtf
[OK] metadata.tsv
[OK] 1mo_Rep1
[OK] 1mo_Rep2
[OK] 1mo_Rep3
[OK] 4mo_Rep1
[OK] 4mo_Rep2
[OK] 4mo_Rep3

All input files found.


In [49]:
# Validate sample metadata
metadata = pd.read_csv(METADATA_FILE, sep=r"\s+")

# Check required columns
required_columns = {"Run", "Age"}

if not required_columns.issubset(metadata.columns):
    raise ValueError("Missing required metadata columns.")

# Check sample identifiers
if len(metadata) != len(SAMPLES):
    raise ValueError("Unexpected number of samples.")

if set(metadata["Run"]) != set(SAMPLES):
    raise ValueError("Sample identifiers do not match.")

# Check experimental groups
expected_groups = {"1_Month": 3, "4_Month": 3,}

if metadata["Age"].value_counts().to_dict() != expected_groups:
    raise ValueError("Unexpected experimental groups.")

# Compare with the sample metadata defined in Section 03
condition_mapping = {"1_Month": "1mo", "4_Month": "4mo",}
course_metadata = metadata[["Run", "Age"]].rename(columns={"Run": "sampleID", "Age": "condition"})
course_metadata["condition"] = (course_metadata["condition"].map(condition_mapping))

course_metadata = (
    course_metadata.set_index("sampleID")
    .loc[SAMPLES]
    .reset_index()
)

if not course_metadata.equals(sample_metadata):
    raise ValueError("Sample metadata do not match.")

display(metadata)
print("\nSample metadata validated successfully.")

,Run,Age
0,1mo_Rep1,1_Month
1,1mo_Rep2,1_Month
2,1mo_Rep3,1_Month
3,4mo_Rep1,4_Month
4,4mo_Rep2,4_Month
5,4mo_Rep3,4_Month



Sample metadata validated successfully.


In [50]:
# Count unique genes in the provided GENCODE annotation
reference_gene_ids = set()
gene_features = 0

with GENCODE_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9:
            continue

        if fields[2] == "gene":
            gene_features += 1

        match = re.search(r'gene_id "([^"]+)"',fields[8])

        if match:
            reference_gene_ids.add(match.group(1))

print("GENCODE gene features:", gene_features)
print("Unique GENCODE gene IDs:", len(reference_gene_ids))

GENCODE gene features: 166
Unique GENCODE gene IDs: 167


## 05. STAR Genome Indexing

Before aligning the RNA-seq reads, a genome index is generated using STAR.

The index is built from the provided mouse chromosome 7 reference sequence (`region.fa`), which contains approximately 3.4 million bases.

The following indexing parameters are used:

- `--genomeSAindexNbases 9`: Defines the length of the sequences used for suffix-array pre-indexing. A value of 9 is suitable for the reduced reference genome.
- `--genomeSAsparseD 1`: Sets the suffix-array sampling interval to 1, providing a dense index without sparsification.

The generated index is checked to confirm that the required files are present and that the recorded parameters match the selected configuration.

The same genome index will be used for all six RNA-seq samples to ensure consistent alignment conditions.

In [51]:
# Generate STAR genome index
STAR_INDEX_DIR.mkdir(parents=True, exist_ok=True)

# --genomeSAindexNbases 9: Uses 9-base sequences for pre-indexing.
# --genomeSAsparseD 1: Uses a dense suffix array (no sparsification).

required_index_files = [
    "Genome",
    "SA",
    "SAindex",
    "genomeParameters.txt"]

index_complete = all(
    (STAR_INDEX_DIR / filename).is_file()
    and (STAR_INDEX_DIR / filename).stat().st_size > 0
    for filename in required_index_files
)

if index_complete:
    print("[OK] STAR genome index already exists.")
elif any(STAR_INDEX_DIR.iterdir()):
    raise RuntimeError("Incomplete STAR index detected. Inspect the directory.")
else:
    command = [
        "STAR",
        "--runThreadN", str(THREADS),
        "--runMode", "genomeGenerate",
        "--genomeDir", str(STAR_INDEX_DIR),
        "--genomeFastaFiles", str(REFERENCE_FASTA),
        "--genomeSAindexNbases", str(GENOME_SA_INDEX_NBASES),
        "--genomeSAsparseD", str(GENOME_SA_SPARSE_D)    ]

    subprocess.run(command, check=True)
    print("[OK] STAR genome index generated.")

	/usr/lib/rna-star/bin/STAR-avx2 --runThreadN 2 --runMode genomeGenerate --genomeDir /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/star_index --genomeFastaFiles /mnt/c/Users/ianca/Documentos/Bioinformatics/transcriptomics-bioinformatics/Module_I/Final_Project/data_raw/region.fa --genomeSAindexNbases 9 --genomeSAsparseD 1
	STAR version: 2.7.11b   compiled: 2026-01-29T20:40:47+00:00 <place not set in Debian package>
Nov 06 15:35:45 ..... started STAR run
Nov 06 15:35:45 ... starting to generate Genome files
Nov 06 15:35:45 ... starting to sort Suffix Array. This may take a long time...
Nov 06 15:35:45 ... sorting Suffix Array chunks and saving them to disk...


!!!!! WARNING: Could not move Log.out file from ./Log.out into /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/STAR/star_index/Log.out. Will keep ./Log.out



Nov 06 15:35:46 ... loading chunks from disk, packing SA...
Nov 06 15:35:46 ... finished generating suffix array
Nov 06 15:35:46 ... generating Suffix Array index
Nov 06 15:35:46 ... completed Suffix Array index
Nov 06 15:35:46 ... writing Genome to disk ...
Nov 06 15:35:46 ... writing Suffix Array to disk ...
Nov 06 15:35:46 ... writing SAindex to disk
Nov 06 15:35:46 ..... finished successfully
[OK] STAR genome index generated.


In [52]:
# Verify STAR genome index files and parameters
for filename in required_index_files:
    path = STAR_INDEX_DIR / filename
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(path)
    print(f"[OK] {filename}")

# Read genome indexing parameters
parameters_file = STAR_INDEX_DIR / "genomeParameters.txt"
parameters = {}

with parameters_file.open() as file:
    for line in file:
        fields = line.strip().split(maxsplit=1)
        if len(fields) == 2:
            parameters[fields[0]] = fields[1]

# Verify selected parameters
expected_parameters = {
    "genomeSAindexNbases": GENOME_SA_INDEX_NBASES,
    "genomeSAsparseD": GENOME_SA_SPARSE_D
}

for name, expected_value in expected_parameters.items():
    observed_value = int(parameters[name])
    print(f"{name}: {observed_value}")
    if observed_value != expected_value:
        raise ValueError(
            f"{name}: expected {expected_value}, "
            f"found {observed_value}")

print("\nSTAR genome index validation completed.")

[OK] Genome
[OK] SA
[OK] SAindex
[OK] genomeParameters.txt
genomeSAindexNbases: 9
genomeSAsparseD: 1

STAR genome index validation completed.


## 06. STAR Read Alignment

- STAR is used to align the RNA-seq reads from six mouse hippocampus samples to the indexed reference genome.
- Each sample is aligned separately using the same genome index and alignment settings. Compressed FASTQ files are read directly using `zcat`, and the resulting alignments are saved as coordinate-sorted BAM files.
- The `--outSAMstrandField intronMotif` option is used to include strand information for spliced alignments, which is relevant for downstream transcript analysis.
- Temporary files are stored in the Linux filesystem to avoid compatibility issues with Windows-mounted directories in WSL.
- After alignment, the BAM files and STAR logs are checked to confirm successful processing.

In [55]:
# Align RNA-seq reads using STAR
STAR_ALIGNMENT_DIR.mkdir(parents=True, exist_ok=True)

# Store temporary files on the Linux filesystem
STAR_TMP_DIR = Path.home() / "star_tmp_final_project"
STAR_TMP_DIR.mkdir(parents=True, exist_ok=True)

STAR_BAMS = {}

for sample in SAMPLES:
    input_fastq = FASTQ_FILES[sample]
    sample_dir = STAR_ALIGNMENT_DIR / sample

    output_bam = sample_dir / "Aligned.sortedByCoord.out.bam"
    output_log = sample_dir / "Log.final.out"

    # Reuse existing alignments only after validation
    if output_bam.is_file() and output_log.is_file():
        subprocess.run(["samtools", "quickcheck", "-v", str(output_bam)], check=True)
        log_text = (sample_dir / "Log.out").read_text(
            errors="replace"
        ) if (sample_dir / "Log.out").is_file() else ""

        if "outSAMstrandField" not in log_text or "intronMotif" not in log_text:
            raise RuntimeError(
                f"{sample}: existing BAM was not verified "
                "with --outSAMstrandField intronMotif."
            )

        print(f"[OK] Existing alignment: {sample}")
        STAR_BAMS[sample] = output_bam
        continue

    # Protect partially generated results
    if sample_dir.exists() and any(sample_dir.iterdir()):
        raise FileExistsError(f"Incomplete STAR outputs: {sample_dir}")

    sample_dir.mkdir(parents=True, exist_ok=True)
    temp_dir = STAR_TMP_DIR / f"{sample}_STARtmp"

    if temp_dir.exists():
        raise FileExistsError(f"Existing STAR temporary directory: {temp_dir}")
    print(f"Aligning {sample}...", flush=True)

    command = [
        "STAR",
        "--runThreadN", str(THREADS),
        "--genomeDir", str(STAR_INDEX_DIR),
        "--readFilesIn", str(input_fastq),
        "--readFilesCommand", "zcat",
        "--outSAMtype", "BAM", "SortedByCoordinate",
        "--outSAMstrandField", "intronMotif",
        "--outTmpDir", str(temp_dir),
        "--outFileNamePrefix", str(sample_dir) + "/"
    ]

    subprocess.run(command, check=True)
    subprocess.run(
        ["samtools", "quickcheck", "-v", str(output_bam)],
        check=True)

    if not output_log.is_file():
        raise FileNotFoundError(output_log)

    STAR_BAMS[sample] = output_bam
    print(f"[OK] Alignment completed: {sample}")

print("\nAll six STAR alignments are available.")

[OK] Existing alignment: 1mo_Rep1
[OK] Existing alignment: 1mo_Rep2
[OK] Existing alignment: 1mo_Rep3
[OK] Existing alignment: 4mo_Rep1
[OK] Existing alignment: 4mo_Rep2
[OK] Existing alignment: 4mo_Rep3

All six STAR alignments are available.


In [56]:
# Verify STAR alignment outputs
for sample in SAMPLES:
    bam = STAR_BAMS[sample]
    log_file = bam.parent / "Log.final.out"
    if not bam.is_file():
        raise FileNotFoundError(bam)
    if not log_file.is_file():
        raise FileNotFoundError(log_file)
    subprocess.run(["samtools", "quickcheck", "-v", str(bam)], check=True)

    print(f"[OK] {sample}")
    print(f"     BAM: {bam.name}")
    print(f"     STAR log: {log_file.name}")

print("\nAll six STAR outputs verified.")

[OK] 1mo_Rep1
     BAM: Aligned.sortedByCoord.out.bam
     STAR log: Log.final.out
[OK] 1mo_Rep2
     BAM: Aligned.sortedByCoord.out.bam
     STAR log: Log.final.out
[OK] 1mo_Rep3
     BAM: Aligned.sortedByCoord.out.bam
     STAR log: Log.final.out
[OK] 4mo_Rep1
     BAM: Aligned.sortedByCoord.out.bam
     STAR log: Log.final.out
[OK] 4mo_Rep2
     BAM: Aligned.sortedByCoord.out.bam
     STAR log: Log.final.out
[OK] 4mo_Rep3
     BAM: Aligned.sortedByCoord.out.bam
     STAR log: Log.final.out

All six STAR outputs verified.


## 07. Alignment Quality Assessment

- After read alignment, STAR generates a `Log.final.out` file containing alignment statistics for each sample.
- These reports are used to examine the number of input reads, uniquely mapped reads, reads mapped to multiple loci, and reads that could not be aligned.
- SAMtools is used to count alignment records in the BAM files.
- Alignment statistics are summarized across all six samples to assess consistency between biological replicates and identify potential differences in mapping performance.
- The number of alignment records in a BAM file is distinguished from the number of mapped reads, since a read may have multiple alignment records.

In [57]:
# Summarize STAR alignment statistics
metrics = {
    "Number of input reads": "Input reads",
    "Uniquely mapped reads number": "Uniquely mapped",
    "Number of reads mapped to multiple loci": "Multimapped",
    "Number of reads unmapped: too short": "Unmapped: too short",
    "Uniquely mapped reads %": "Uniquely mapped (%)"}
alignment_results = []

for sample in SAMPLES:
    log_file = (STAR_ALIGNMENT_DIR / sample / "Log.final.out")

    if not log_file.is_file():
        raise FileNotFoundError(log_file)
    log_data = {}

    with log_file.open() as file:
        for line in file:
            if "|" in line:
                key, value = line.split("|", 1)
                log_data[key.strip()] = value.strip()
    row = {"Sample": sample}

    for star_key, label in metrics.items():
        row[label] = log_data.get(star_key, "NA")
    alignment_results.append(row)

alignment_summary = pd.DataFrame(alignment_results)
display(alignment_summary)

,Sample,Input reads,Uniquely mapped,Multimapped,Unmapped: too short,Uniquely mapped (%)
0,1mo_Rep1,32447,31921,521,1,98.38%
1,1mo_Rep2,40653,40076,570,1,98.58%
2,1mo_Rep3,40042,39427,600,1,98.46%
3,4mo_Rep1,35269,34684,581,0,98.34%
4,4mo_Rep2,37649,37043,599,0,98.39%
5,4mo_Rep3,41626,40916,705,1,98.29%


In [58]:
# Count BAM alignment records using SAMtools
bam_counts = []

for sample in SAMPLES:
    bam = STAR_BAMS[sample]
    result = subprocess.run(
        ["samtools", "view", "-c", str(bam)],
        capture_output=True,
        text=True,
        check=True)
    bam_counts.append({
        "Sample": sample,
        "BAM alignment records": int(result.stdout.strip())
    })

bam_summary = pd.DataFrame(bam_counts)
display(bam_summary)

,Sample,BAM alignment records
0,1mo_Rep1,33086
1,1mo_Rep2,41320
2,1mo_Rep3,40776
3,4mo_Rep1,35948
4,4mo_Rep2,38360
5,4mo_Rep3,42488


In [59]:
# Inspect alignment statistics for 1mo_Rep2
sample = "1mo_Rep2"
log_file = STAR_ALIGNMENT_DIR / sample / "Log.final.out"

log_data = {}

with log_file.open() as file:
    for line in file:
        if "|" in line:
            key, value = line.split("|", 1)
            log_data[key.strip()] = value.strip()

unique_reads = int(log_data["Uniquely mapped reads number"])
multimapped_reads = int(log_data["Number of reads mapped to multiple loci"])
too_short = int(log_data["Number of reads unmapped: too short"])
mapped_reads = unique_reads + multimapped_reads

print("Sample:", sample)
print("Mapped reads:", mapped_reads)
print("Uniquely mapped reads:", unique_reads)
print("Multimapped reads:", multimapped_reads)
print("Unmapped reads (too short):", too_short)

# Count BAM alignment records independently
result = subprocess.run(
    ["samtools", "view", "-c", str(STAR_BAMS[sample])],
    capture_output=True,
    text=True,
    check=True
)

print("BAM alignment records:", result.stdout.strip())

Sample: 1mo_Rep2
Mapped reads: 40646
Uniquely mapped reads: 40076
Multimapped reads: 570
Unmapped reads (too short): 1
BAM alignment records: 41320


## 08. Transcript Assembly — PsiCLASS

- PsiCLASS is used to reconstruct transcripts from the STAR-aligned RNA-seq reads.
- The six coordinate-sorted BAM files are provided through a `bamlist.psiclass` file, preserving the sample order defined in the project configuration.
- PsiCLASS generates transcript assemblies for individual samples and a consensus assembly representing transcripts supported across the dataset.
- The resulting GTF files are checked for completeness before proceeding to gene annotation.

In [60]:
# Prepare BAM files for PsiCLASS
PSICLASS_PREFIX = PSICLASS_DIR / "psiclass"
PSICLASS_VOTE = PSICLASS_DIR / "psiclass_vote.gtf"
BAMLIST = PSICLASS_DIR / "bamlist.psiclass"
PSICLASS_EXE = (Path.home() / "miniconda3/envs/psiclass/bin/psiclass")

# Collect BAM files in sample order
BAM_FILES = [STAR_BAMS[sample] for sample in SAMPLES]

for sample, bam in zip(SAMPLES, BAM_FILES):
    if not bam.is_file():
        raise FileNotFoundError(bam)
    subprocess.run(
        ["samtools", "quickcheck", "-v", str(bam)],
        check=True)
    print(f"[OK] {sample}")

# Prepare PsiCLASS directory
PSICLASS_DIR.mkdir(parents=True, exist_ok=True)

# Write the BAM list
bamlist_content = "\n".join(
    str(bam.resolve()) for bam in BAM_FILES) + "\n"

if BAMLIST.exists():
    if BAMLIST.read_text() != bamlist_content:
        raise ValueError("Existing BAM list differs from current samples.")
    print("\n[OK] Existing BAM list verified.")
else:
    BAMLIST.write_text(bamlist_content)
    print("\n[OK] BAM list created.")

if not PSICLASS_EXE.is_file():
    raise FileNotFoundError(PSICLASS_EXE)

print("\nPsiCLASS sample order:")

for i, sample in enumerate(SAMPLES):
    print(f"{i}: {sample}")

[OK] 1mo_Rep1
[OK] 1mo_Rep2
[OK] 1mo_Rep3
[OK] 4mo_Rep1
[OK] 4mo_Rep2
[OK] 4mo_Rep3

[OK] BAM list created.

PsiCLASS sample order:
0: 1mo_Rep1
1: 1mo_Rep2
2: 1mo_Rep3
3: 4mo_Rep1
4: 4mo_Rep2
5: 4mo_Rep3


In [61]:
# Run PsiCLASS transcript assembly
PSICLASS_LOG = PSICLASS_DIR / "psiclass.log"

# Define expected output files
expected_gtfs = [PSICLASS_VOTE] + [
    PSICLASS_DIR / f"psiclass_sample_{i}.gtf"
    for i in range(len(SAMPLES))]

# Check whether the assembly is already complete
assembly_complete = all(
    path.is_file() and path.stat().st_size > 0
    for path in expected_gtfs)

if assembly_complete:
    print("[OK] PsiCLASS transcript assembly already exists.")

elif any(path.exists() for path in expected_gtfs):
    raise FileExistsError("Partial PsiCLASS assembly detected. Inspect existing files.")
elif PSICLASS_LOG.exists():
    raise FileExistsError("An existing PsiCLASS log was found. Inspect it before rerunning.")
else:
    command = [
        str(PSICLASS_EXE),
        "--lb", str(BAMLIST),
        "-o", str(PSICLASS_PREFIX),
        "-p", str(THREADS)]
    print("Running PsiCLASS transcript assembly...", flush=True)

    with PSICLASS_LOG.open("x") as log:
        subprocess.run(
            command,
            stdout=log,
            stderr=subprocess.STDOUT,
            check=True)
    print("[OK] PsiCLASS execution completed.")

# Verify all expected outputs
for path in expected_gtfs:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(path)
    print(f"[OK] {path.name}")

print("\nAll PsiCLASS transcript assemblies are available.")

Running PsiCLASS transcript assembly...
[OK] PsiCLASS execution completed.
[OK] psiclass_vote.gtf
[OK] psiclass_sample_0.gtf
[OK] psiclass_sample_1.gtf
[OK] psiclass_sample_2.gtf
[OK] psiclass_sample_3.gtf
[OK] psiclass_sample_4.gtf
[OK] psiclass_sample_5.gtf

All PsiCLASS transcript assemblies are available.


In [62]:
# Summarize PsiCLASS transcript assemblies
assembly_summary = []
gtf_files = [("Consensus", PSICLASS_VOTE)] + [
    (
        sample,
        PSICLASS_DIR / f"psiclass_sample_{i}.gtf"
    )
    for i, sample in enumerate(SAMPLES)
]

for sample, gtf_file in gtf_files:
    gene_ids = set()
    transcript_ids = set()
    exon_counts = {}

    with gtf_file.open() as gtf:
        for line in gtf:
            if line.startswith("#"):
                continue

            fields = line.rstrip("\n").split("\t")

            if len(fields) != 9:
                continue

            attrs = dict(re.findall(r'(\w+)\s+"([^"]+)"', fields[8]))
            gene_id = attrs.get("gene_id")
            transcript_id = attrs.get("transcript_id")

            if gene_id:
                gene_ids.add(gene_id)
            if transcript_id:
                transcript_ids.add(transcript_id)
                if fields[2] == "exon":
                    exon_counts[transcript_id] = (exon_counts.get(transcript_id, 0) + 1)

    assembly_summary.append({
        "Assembly": sample,
        "Genes (loci)": len(gene_ids),
        "Transcripts": len(transcript_ids),
        "Single-exon transcripts": sum(
            count == 1 for count in exon_counts.values()
        )
    })

assembly_df = pd.DataFrame(assembly_summary)
display(assembly_df)

,Assembly,Genes (loci),Transcripts,Single-exon transcripts
0,Consensus,168,211,84
1,1mo_Rep1,173,204,85
2,1mo_Rep2,171,210,84
3,1mo_Rep3,173,212,86
4,4mo_Rep1,173,209,85
5,4mo_Rep2,172,211,84
6,4mo_Rep3,173,212,86


## 09. Gene Annotation

- The PsiCLASS transcript assemblies are annotated using the GENCODE vM30 reference annotation.
- The `add-genename` utility is used to associate assembled transcripts with reference gene names, while `add-strand` assigns genomic strand information.
- The consensus and six sample-specific assemblies are processed using the original course annotation workflow.
- Annotated GTF files are stored in a separate `WithGeneNames` directory, preserving the original PsiCLASS assemblies.
- The final strand-annotated consensus GTF will be used for differential expression analysis with Cuffdiff2 and gene-level read counting for DESeq2.

In [63]:
# Prepare PsiCLASS annotation inputs
ADD_GENENAME = (Path.home() / "miniconda3/envs/psiclass/bin/add-genename")
ADD_STRAND = REPO_DIR.parent / "docs" / "add-strand"

ANNOTATION_DIR = PSICLASS_DIR / "WithGeneNames"
ANNOTATION_LIST = PSICLASS_DIR / "psiclass_annotation.list"

# Collect consensus and sample-specific assemblies
annotation_gtfs = [PSICLASS_VOTE] + [
    PSICLASS_DIR / f"psiclass_sample_{i}.gtf"
    for i in range(len(SAMPLES))
]

# Verify annotation inputs and utilities
for path in [
    GENCODE_GTF,
    ADD_GENENAME,
    ADD_STRAND,
    *annotation_gtfs
]:
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f"[OK] {path.name}")

# Prepare annotation file list
list_content = "\n".join(
    str(path.resolve()) for path in annotation_gtfs
) + "\n"

if ANNOTATION_LIST.exists():
    if ANNOTATION_LIST.read_text() != list_content:
        raise ValueError("Existing annotation list differs.")
    print("\n[OK] Annotation list verified.")
else:
    ANNOTATION_LIST.write_text(list_content)
    print("\n[OK] Annotation list created.")

ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
print("Assemblies:", len(annotation_gtfs))

[OK] gencode.vM30.region.gtf
[OK] add-genename
[OK] add-strand
[OK] psiclass_vote.gtf
[OK] psiclass_sample_0.gtf
[OK] psiclass_sample_1.gtf
[OK] psiclass_sample_2.gtf
[OK] psiclass_sample_3.gtf
[OK] psiclass_sample_4.gtf
[OK] psiclass_sample_5.gtf

[OK] Annotation list created.
Assemblies: 7


In [64]:
# Assign GENCODE gene names to PsiCLASS assemblies
# Define expected annotation outputs
expected_annotations = [
    ANNOTATION_DIR / gtf.name
    for gtf in annotation_gtfs]

# Check whether all annotations already exist
annotations_complete = all(
    path.is_file() and path.stat().st_size > 0
    for path in expected_annotations)

if annotations_complete:
    print("[OK] Gene annotations already exist.")
elif any(path.exists() for path in expected_annotations):
    raise FileExistsError(
        "Partial annotation outputs detected. "
        "Inspect existing files before rerunning.")
else:
    command = [
        str(ADD_GENENAME),
        str(GENCODE_GTF),
        str(ANNOTATION_LIST),
        "-o", str(ANNOTATION_DIR)]

    print("Assigning GENCODE gene names...", flush=True)
    subprocess.run(command, check=True)

# Verify annotated GTF files
for path in expected_annotations:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(path)
    print(f"[OK] {path.name}")

# Define annotated consensus file
ANNOTATED_VOTE = ANNOTATION_DIR / "psiclass_vote.gtf"
print("\nGENCODE gene-name annotation completed.")

Assigning GENCODE gene names...
[OK] psiclass_vote.gtf
[OK] psiclass_sample_0.gtf
[OK] psiclass_sample_1.gtf
[OK] psiclass_sample_2.gtf
[OK] psiclass_sample_3.gtf
[OK] psiclass_sample_4.gtf
[OK] psiclass_sample_5.gtf

GENCODE gene-name annotation completed.


In [65]:
# Assign genomic strand information to the PsiCLASS consensus
FINAL_GTF = ANNOTATION_DIR / "psiclass_vote.withStrand.gtf"

# Verify input files
for path in [GENCODE_GTF, ADD_STRAND, ANNOTATED_VOTE]:
    if not path.is_file():
        raise FileNotFoundError(path)

# Reuse an existing annotation
if FINAL_GTF.is_file() and FINAL_GTF.stat().st_size > 0:
    print("[OK] Strand-annotated GTF already exists.")
elif FINAL_GTF.exists():
    raise FileExistsError(f"Empty or incomplete annotation: {FINAL_GTF}")
else:
    command = [
        "perl",
        str(ADD_STRAND),
        str(GENCODE_GTF),
        "-r"]

    # Write to a temporary file before saving the final annotation
    temp_gtf = ANNOTATION_DIR / "psiclass_vote.withStrand.gtf.tmp"

    if temp_gtf.exists():
        raise FileExistsError(temp_gtf)
    print("Assigning genomic strand information...", flush=True)

    with ANNOTATED_VOTE.open("r") as source:
        with temp_gtf.open("x") as output:
            subprocess.run(
                command,
                stdin=source,
                stdout=output,
                check=True)

    if temp_gtf.stat().st_size == 0:
        raise ValueError("Strand annotation output is empty.")

    temp_gtf.rename(FINAL_GTF)
    print("[OK] Genomic strand annotation completed.")
print("Final annotation:", FINAL_GTF)

Assigning genomic strand information...
[OK] Genomic strand annotation completed.
Final annotation: /mnt/d/Bioinformatics/transcriptomics-data/Module_I/Final_Project/results/PsiCLASS/WithGeneNames/psiclass_vote.withStrand.gtf


In [67]:
# Verify final PsiCLASS annotation
gene_ids = set()
transcript_ids = set()
strand_counts = {"+": 0, "-": 0, ".": 0}
records = 0

with FINAL_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#") or not line.strip():
            continue
        fields = line.rstrip("\n").split("\t")
        if len(fields) != 9:
            raise ValueError("Invalid GTF record.")
        
        records += 1
        strand = fields[6]

        if strand not in strand_counts:
            raise ValueError(f"Unexpected strand: {strand}")

        strand_counts[strand] += 1
        attrs = dict(re.findall(r'(\w+)\s+"([^"]+)"', fields[8]))
        if attrs.get("gene_id"):
            gene_ids.add(attrs["gene_id"])
        if attrs.get("transcript_id"):
            transcript_ids.add(attrs["transcript_id"])

print("GTF records:", records)
print("Genes (loci):", len(gene_ids))
print("Transcripts:", len(transcript_ids))
print("Positive strand:", strand_counts["+"])
print("Negative strand:", strand_counts["-"])
print("Unknown strand:", strand_counts["."])

GTF records: 1155
Genes (loci): 142
Transcripts: 185
Positive strand: 463
Negative strand: 692
Unknown strand: 0


## 10. Differential Gene Expression — Cuffdiff2

- Cuffdiff2 is used to compare gene expression between 1-month-old and 4-month-old mouse hippocampus samples, with three biological replicates per condition.
- The analysis uses the STAR-aligned BAM files and the strand-annotated PsiCLASS consensus assembly.
- Cuffdiff2 estimates gene expression in fragments per kilobase of transcript per million mapped fragments (FPKM) and performs statistical tests to identify differentially expressed genes.
- The resulting `gene_exp.diff` file contains expression estimates, log2 fold changes, p-values, and FDR-adjusted q-values.
- Differential expression is assessed using a significance threshold of 0.05. Additional expression and fold-change thresholds are examined in the results summary.

In [68]:
# Prepare Cuffdiff2 input files
# Define biological replicate groups
control_bams = [STAR_BAMS[sample] for sample in SAMPLES_1MO]
test_bams = [STAR_BAMS[sample] for sample in SAMPLES_4MO]

# Verify final PsiCLASS annotation
if not FINAL_GTF.is_file() or FINAL_GTF.stat().st_size == 0:
    raise FileNotFoundError(FINAL_GTF)

print("[OK] Final PsiCLASS annotation")

# Verify BAM files
for sample, bam in zip(SAMPLES, control_bams + test_bams):
    if not bam.is_file():
        raise FileNotFoundError(bam)

    subprocess.run(["samtools", "quickcheck", "-v", str(bam)], check=True)

    print(f"[OK] {sample}")

# Prepare replicate groups for Cuffdiff2
control_group = ",".join(map(str, control_bams))
test_group = ",".join(map(str, test_bams))

print("\nControl samples:", len(control_bams))
print("Test samples:", len(test_bams))
print("All Cuffdiff2 input files verified.")

[OK] Final PsiCLASS annotation
[OK] 1mo_Rep1
[OK] 1mo_Rep2
[OK] 1mo_Rep3
[OK] 4mo_Rep1
[OK] 4mo_Rep2
[OK] 4mo_Rep3

Control samples: 3
Test samples: 3
All Cuffdiff2 input files verified.


In [69]:
# Run Cuffdiff2 differential expression analysis
CUFFDIFF_LOG = CUFFDIFF_DIR / "cuffdiff.log"
GENE_DIFF = CUFFDIFF_DIR / "gene_exp.diff"

# Check whether the main results already exist
expected_outputs = [GENE_DIFF, CUFFDIFF_DIR / "genes.fpkm_tracking"]
results_complete = all(
    path.is_file() and path.stat().st_size > 0
    for path in expected_outputs
)

if results_complete:
    print("[OK] Cuffdiff2 results already exist.")
elif CUFFDIFF_DIR.exists() and any(CUFFDIFF_DIR.iterdir()):
    raise FileExistsError(
        "Existing Cuffdiff2 files detected. "
        "Inspect the directory before rerunning.")
else:
    CUFFDIFF_DIR.mkdir(parents=True, exist_ok=True)
    command = [
        "cuffdiff",
        "-o", str(CUFFDIFF_DIR),
        "-p", str(THREADS),
        "-L", "1mo,4mo",
        str(FINAL_GTF),
        control_group,
        test_group]
    print("Running Cuffdiff2...", flush=True)

    with CUFFDIFF_LOG.open("x") as log:
        result = subprocess.run(
            command,
            stdout=log,
            stderr=subprocess.STDOUT)

    if result.returncode != 0:
        print(CUFFDIFF_LOG.read_text(errors="replace")[-3000:])
        raise RuntimeError("Cuffdiff2 analysis failed.")
    print("[OK] Cuffdiff2 execution completed.")

# Verify output files
for path in expected_outputs:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(path)
    print(f"[OK] {path.name}")
print("\nCuffdiff2 analysis completed.")

Running Cuffdiff2...
[OK] Cuffdiff2 execution completed.
[OK] gene_exp.diff
[OK] genes.fpkm_tracking

Cuffdiff2 analysis completed.


In [70]:
# Summarize Cuffdiff2 differential expression results
# Load gene-level results
cuff_results = pd.read_csv(GENE_DIFF, sep="\t", dtype={"gene_id": str})

# Convert statistical and expression columns
numeric_columns = [
    "p_value",
    "q_value",
    "log2(fold_change)",
    "value_1",
    "value_2"
]

for column in numeric_columns:
    cuff_results[column] = pd.to_numeric(cuff_results[column], errors="coerce")

# Identify FDR-significant entries
significant = (
    cuff_results["significant"]
    .astype(str)
    .str.lower()
    .eq("yes"))

# Apply p-value and fold-change thresholds
pval_fc = (
    cuff_results["status"].eq("OK")
    & cuff_results["p_value"].le(0.05)
    & cuff_results["log2(fold_change)"].abs().ge(1))

# Identify expressed entries (FPKM >= 2 in either condition)
expressed = (
    cuff_results["value_1"].ge(2)
    | cuff_results["value_2"].ge(2))

# Summarize results
cuff_summary = pd.DataFrame({
    "Metric": [
        "Total gene-level entries",
        "Significant entries (FDR <= 0.05)",
        "p-value <= 0.05 and |log2FC| >= 1",
        "FPKM >= 2 in at least one condition"
    ],
    "Count": [
        len(cuff_results),
        int(significant.sum()),
        int(pval_fc.sum()),
        int(expressed.sum())
    ]
})

display(cuff_summary)

,Metric,Count
0,Total gene-level entries,142
1,Significant entries (FDR <= 0.05),4
2,p-value <= 0.05 and |log2FC| >= 1,1
3,FPKM >= 2 in at least one condition,142


In [71]:
# Identify reference GENCODE genes among significant Cuffdiff2 features
from collections import defaultdict

# Map PsiCLASS loci to reference gene names
gene_mapping = defaultdict(set)

with FINAL_GTF.open() as gtf:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9 or fields[2] != "transcript":
            continue

        attrs = dict(re.findall(r'(\w+)\s+"([^"]+)"', fields[8]))
        gene_id = attrs.get("gene_id")
        gene_name = attrs.get("gene_name")

        if gene_id and gene_name and "novel" not in gene_name.lower():
            gene_mapping[gene_id].add(gene_name)

# Select significant Cuffdiff2 features
significant_ids = cuff_results.loc[
    cuff_results["significant"]
    .astype(str)
    .str.lower()
    .eq("yes"),
    "gene_id"
].astype(str).unique()

# Identify distinct reference gene names
reference_genes = set()

for gene_id in significant_ids:
    names = gene_mapping.get(gene_id, set())
    reference_genes.update(names)

    print(
        f"{gene_id}: "
        f"{', '.join(sorted(names)) or 'No reference annotation'}")

print("\nDistinct GENCODE gene names:", len(reference_genes))

7.21: Sirt2
7.395: Rps12-ps4
7.422: Mag
7.9: Sirt2

Distinct GENCODE gene names: 3


## 11. Gene-Level Read Counting

- Gene-level read counts are generated for differential expression analysis using DESeq2.
- The final strand-annotated PsiCLASS consensus assembly is used to define gene features. Exons are grouped by gene using `txdbmaker` and `GenomicFeatures`.
- Before counting, the STAR-aligned BAM files are sorted by read name using SAMtools.
- Read counting is performed with `GenomicAlignments::summarizeOverlaps()` using the Union mode, following the counting parameters from the course workflow.
- The resulting matrix contains raw gene-level counts for all six RNA-seq samples. These counts will be used as input for DESeq2.

In [72]:
# Prepare name-sorted BAM files for DESeq2
SORTED_DIR.mkdir(parents=True, exist_ok=True)
SORTED_BAMS = []

for sample in SAMPLES:
    input_bam = STAR_BAMS[sample]
    output_bam = SORTED_DIR / f"{sample}.so.bam"

    if not input_bam.is_file():
        raise FileNotFoundError(input_bam)

    # Reuse existing BAM files after validation
    if output_bam.is_file():
        subprocess.run(["samtools", "quickcheck", "-v", str(output_bam)], check=True)
        print(f"[OK] Existing sorted BAM: {sample}")
    else:
        print(f"Sorting BAM: {sample}", flush=True)

        subprocess.run([
            "samtools", "sort",
            "-n",
            "-@", str(THREADS),
            "-O", "BAM",
            "-o", str(output_bam),
            str(input_bam)
        ], check=True)

        subprocess.run(["samtools", "quickcheck", "-v", str(output_bam)], check=True)

        print(f"[OK] Sorted: {sample}")

    SORTED_BAMS.append(output_bam)
print("\nAll six name-sorted BAM files are available.")

Sorting BAM: 1mo_Rep1


[bam_sort_core] merging from 0 files and 2 in-memory blocks...


[OK] Sorted: 1mo_Rep1
Sorting BAM: 1mo_Rep2


[bam_sort_core] merging from 0 files and 2 in-memory blocks...


[OK] Sorted: 1mo_Rep2
Sorting BAM: 1mo_Rep3


[bam_sort_core] merging from 0 files and 2 in-memory blocks...


[OK] Sorted: 1mo_Rep3
Sorting BAM: 4mo_Rep1
[OK] Sorted: 4mo_Rep1
Sorting BAM: 4mo_Rep2


[bam_sort_core] merging from 0 files and 2 in-memory blocks...
[bam_sort_core] merging from 0 files and 2 in-memory blocks...


[OK] Sorted: 4mo_Rep2
Sorting BAM: 4mo_Rep3
[OK] Sorted: 4mo_Rep3

All six name-sorted BAM files are available.


[bam_sort_core] merging from 0 files and 2 in-memory blocks...


In [73]:
# Verify name-sorted BAM files
for sample, bam in zip(SAMPLES, SORTED_BAMS):
    if not bam.is_file():
        raise FileNotFoundError(bam)

    subprocess.run(["samtools", "quickcheck", "-v", str(bam)], check=True)
    result = subprocess.run(
        ["samtools", "view", "-H", str(bam)],
        capture_output=True,
        text=True,
        check=True)

    if "SO:queryname" not in result.stdout:
        raise ValueError(f"Unexpected BAM sorting order: {sample}")
    print(f"[OK] {sample}: queryname sorted")

print("\nAll six BAM files are ready for gene counting.")

[OK] 1mo_Rep1: queryname sorted
[OK] 1mo_Rep2: queryname sorted
[OK] 1mo_Rep3: queryname sorted
[OK] 4mo_Rep1: queryname sorted
[OK] 4mo_Rep2: queryname sorted
[OK] 4mo_Rep3: queryname sorted

All six BAM files are ready for gene counting.


In [74]:
# Generate gene-level raw counts using R
COUNTS_FILE = (DESEQ2_RESULTS_DIR / "raw_counts.1mo_4mo.psiclass.csv")
DESEQ2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RSCRIPT_EXE = (Path.home() / "miniconda3/envs/transcriptomics/bin/Rscript")

# Verify required inputs
for path in [FINAL_GTF, RSCRIPT_EXE, *SORTED_BAMS]:
    if not path.is_file():
        raise FileNotFoundError(path)

# Pass file paths and sample names to R
r_env = os.environ.copy()
r_env["PROJECT_GTF"] = str(FINAL_GTF)
r_env["PROJECT_BAMS"] = ";".join(map(str, SORTED_BAMS))
r_env["PROJECT_SAMPLES"] = ";".join(SAMPLES)
r_env["PROJECT_COUNTS"] = str(COUNTS_FILE)

r_code = r'''
suppressPackageStartupMessages({
    library(GenomicFeatures)
    library(GenomicAlignments)
    library(Rsamtools)
    library(SummarizedExperiment)})

# Read input paths
gtf_file <- Sys.getenv("PROJECT_GTF")
output_file <- Sys.getenv("PROJECT_COUNTS")

bam_files <- strsplit(
    Sys.getenv("PROJECT_BAMS"), ";", fixed = TRUE
)[[1]]

sample_ids <- strsplit(
    Sys.getenv("PROJECT_SAMPLES"), ";", fixed = TRUE
)[[1]]

# Build transcript database from PsiCLASS annotation
txdb <- txdbmaker::makeTxDbFromGFF(
    gtf_file,
    format = "gtf")

# Group exons by gene
exons_by_gene <- exonsBy(txdb, by = "gene")

cat("Genes in TxDb:", length(exons_by_gene), "\n")

# Open name-sorted BAM files
bam_list <- BamFileList(
    bam_files,
    yieldSize = 100000)

# Count reads using the course parameters
se <- summarizeOverlaps(
    features = exons_by_gene,
    reads = bam_list,
    mode = "Union",
    singleEnd = FALSE,
    ignore.strand = TRUE,
    fragments = TRUE)

# Extract raw count matrix
count_matrix <- assay(se)
colnames(count_matrix) <- sample_ids

# Validate counts
if (anyNA(count_matrix) || any(count_matrix < 0)) {
    stop("Invalid read counts.")}

if (sum(count_matrix) == 0) {stop("No reads assigned to genes.")}

# Export raw counts
write.csv(count_matrix, output_file, row.names = TRUE)

cat("\nGene-level counting completed!\n")
cat("Loci:", nrow(count_matrix), "\n")
cat("Samples:", ncol(count_matrix), "\n")
cat("Total assigned counts:", sum(count_matrix), "\n")
'''

# Preserve existing results
if COUNTS_FILE.is_file() and COUNTS_FILE.stat().st_size > 0:
    print("[OK] Existing gene-level count matrix found.")

elif COUNTS_FILE.exists():
    raise FileExistsError(f"Empty or incomplete count matrix: {COUNTS_FILE}")
else:
    print("Running gene-level read counting...", flush=True)

    result = subprocess.run(
        [str(RSCRIPT_EXE), "--vanilla", "-e", r_code],
        env=r_env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace")

    print(result.stdout)

    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError("Gene-level read counting failed.")

# Inspect count matrix while preserving PsiCLASS gene IDs
counts = pd.read_csv(
    COUNTS_FILE,
    index_col=0,
    dtype=str)

if list(counts.columns) != SAMPLES:
    raise ValueError("Count matrix sample order does not match.")

print("\nRaw count matrix:")
print("Loci:", counts.shape[0])
print("Samples:", counts.shape[1])

display(counts.head())

Running gene-level read counting...
Genes in TxDb: 142 

Gene-level counting completed!
Loci: 142 
Samples: 6 
Total assigned counts: 205593 

Warning message:
package ‘S4Vectors’ was built under R version 4.5.3 
Import genomic features from the file as a GRanges object ... OK
Prepare the 'metadata' data frame ... OK
Make the TxDb object ... OK
Warning message:
In .makeTxDb_normarg_chrominfo(chrominfo) :
  genome version information is not available for this TxDb object


Raw count matrix:
Loci: 142
Samples: 6


,1mo_Rep1,1mo_Rep2,1mo_Rep3,4mo_Rep1,4mo_Rep2,4mo_Rep3
7.100,189,230,255,162,260,224
7.103,10,13,13,6,9,14
7.104,11,6,8,9,12,7
7.105,98,97,106,83,103,106
7.107,44,66,73,95,62,88


## 12. Differential Gene Expression — DESeq2

- DESeq2 is used to identify differentially expressed genes between 1-month-old and 4-month-old mouse hippocampus samples.
- The analysis uses the raw gene-level counts generated from the PsiCLASS consensus annotation, with three biological replicates per condition.
- DESeq2 estimates size factors to account for differences in sequencing depth, models count variability, and performs statistical tests for differential expression.
- The 1-month-old group (`1mo`) is used as the reference condition. Log2 fold changes therefore represent expression changes in the 4-month-old group relative to the 1-month-old group.
- Statistical significance is evaluated using both unadjusted p-values and Benjamini–Hochberg adjusted p-values (FDR), with a threshold of 0.05.
- The analysis exports complete differential expression results, normalized counts, and tables of significant loci.

In [75]:
# Prepare DESeq2 sample metadata
PHENO_FILE = DESEQ2_DIR / "pheno_1mo_4mo.csv"
DESEQ2_DIR.mkdir(parents=True, exist_ok=True)

# Match sample metadata to the count matrix
pheno = (
    sample_metadata
    .set_index("sampleID")
    .loc[SAMPLES]
    .reset_index())

# Verify sample order
if pheno["sampleID"].tolist() != counts.columns.tolist():
    raise ValueError("Sample order does not match the count matrix.")

# Verify experimental groups
expected_groups = {"1mo": len(SAMPLES_1MO), "4mo": len(SAMPLES_4MO)}

if pheno["condition"].value_counts().to_dict() != expected_groups:
    raise ValueError("Unexpected experimental group sizes.")

# Save sample metadata without overwriting existing data
if PHENO_FILE.is_file():
    existing_pheno = pd.read_csv(PHENO_FILE)
    if not existing_pheno.equals(pheno):
        raise ValueError("Existing metadata differs from current metadata.")
    print("[OK] Existing DESeq2 metadata verified.")
else:
    pheno.to_csv(PHENO_FILE, index=False)
    print("[OK] DESeq2 metadata created.")

display(pheno)

[OK] DESeq2 metadata created.


,sampleID,condition
0,1mo_Rep1,1mo
1,1mo_Rep2,1mo
2,1mo_Rep3,1mo
3,4mo_Rep1,4mo
4,4mo_Rep2,4mo
5,4mo_Rep3,4mo


In [76]:
# Run DESeq2 differential expression analysis
RESULTS_FILE = DESEQ2_RESULTS_DIR / "results_1mo_4mo.psiclass.csv"
PVAL_FILE = DESEQ2_RESULTS_DIR / "results_1mo_4mo.pval05.psiclass.csv"
QVAL_FILE = DESEQ2_RESULTS_DIR / "results_1mo_4mo.qval05.psiclass.csv"
NORMALIZED_FILE = DESEQ2_RESULTS_DIR / "normalized_counts.1mo_4mo.psiclass.csv"

output_files = [
    RESULTS_FILE,
    PVAL_FILE,
    QVAL_FILE,
    NORMALIZED_FILE]

# Verify required inputs
for path in [COUNTS_FILE, PHENO_FILE, RSCRIPT_EXE]:
    if not path.is_file():
        raise FileNotFoundError(path)

DESEQ2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Pass input and output paths to R
r_env = os.environ.copy()
r_env["PROJECT_COUNTS"] = str(COUNTS_FILE)
r_env["PROJECT_PHENO"] = str(PHENO_FILE)
r_env["PROJECT_RESULTS_DIR"] = str(DESEQ2_RESULTS_DIR)

r_code = r'''
suppressPackageStartupMessages(library(DESeq2))

# Import raw counts while preserving PsiCLASS gene IDs
counts_df <- read.csv(
    Sys.getenv("PROJECT_COUNTS"),
    check.names = FALSE,
    colClasses = "character"
)

gene_ids <- counts_df[[1]]

if (anyNA(gene_ids) || any(gene_ids == "") ||
    anyDuplicated(gene_ids)) {
    stop("Invalid PsiCLASS gene identifiers.")}

counts <- as.matrix(counts_df[, -1, drop = FALSE])
rownames(counts) <- gene_ids
storage.mode(counts) <- "numeric"

if (anyNA(counts) || any(counts < 0) ||
    any(counts != round(counts))) {
    stop("Counts must be non-negative integers.")}

storage.mode(counts) <- "integer"

# Import sample metadata
metadata <- read.csv(Sys.getenv("PROJECT_PHENO"), row.names = 1)

if (!setequal(colnames(counts), rownames(metadata))) {
    stop("Sample names do not match.")}

metadata <- metadata[colnames(counts), , drop = FALSE]

# Set the reference group
metadata$condition <- factor(metadata$condition, levels = c("1mo", "4mo"))

if (anyNA(metadata$condition)) {
    stop("Invalid experimental conditions.")}

# Fit the differential expression model
dds <- DESeqDataSetFromMatrix(
    countData = counts,
    colData = metadata,
    design = ~ condition)

dds <- DESeq(dds)

# Extract results for 4mo versus 1mo
res <- results(dds, contrast = c("condition", "4mo", "1mo"))

res_df <- as.data.frame(res)
outdir <- Sys.getenv("PROJECT_RESULTS_DIR")

# Export complete DESeq2 results
write.csv(res_df, file.path(outdir, "results_1mo_4mo.psiclass.csv"))

# Export normalized counts
write.csv(counts(dds, normalized = TRUE), file.path(outdir, "normalized_counts.1mo_4mo.psiclass.csv"))

# Select loci with unadjusted p-value <= 0.05
res_pval <- res_df[
    !is.na(res_df$pvalue) & res_df$pvalue <= 0.05,
    ,
    drop = FALSE]

write.csv(res_pval, file.path(outdir, "results_1mo_4mo.pval05.psiclass.csv"))

# Select loci with FDR-adjusted p-value <= 0.05
res_qval <- res_df[
    !is.na(res_df$padj) & res_df$padj <= 0.05,
    ,
    drop = FALSE
]

write.csv(res_qval, file.path(outdir, "results_1mo_4mo.qval05.psiclass.csv"))

# Summarize differential expression results
cat("\nDESeq2 analysis completed!\n")
cat("Loci:", nrow(res_df), "\n")
cat("Samples:", ncol(counts), "\n")
cat("p-value <= 0.05:", nrow(res_pval), "\n")
cat("padj <= 0.05:", nrow(res_qval), "\n")
cat(
    "|log2FoldChange| >= 2:",
    sum(abs(res_df$log2FoldChange) >= 2, na.rm = TRUE),
    "\n"
)
'''

# Reuse existing results without overwriting them
if all(
    path.is_file() and path.stat().st_size > 0
    for path in output_files
):
    print("[OK] Existing DESeq2 results found.")
elif any(path.exists() for path in output_files):
    raise FileExistsError("Partial DESeq2 results detected. Inspect before rerunning.")
else:
    print("Running DESeq2...", flush=True)

    result = subprocess.run(
        [str(RSCRIPT_EXE), "--vanilla", "-e", r_code],
        env=r_env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace")

    print(result.stdout)

    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError("DESeq2 analysis failed.")

# Verify generated files
for path in output_files:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(path)

    print(f"[OK] {path.name}")

Running DESeq2...

DESeq2 analysis completed!
Loci: 142 
Samples: 6 
p-value <= 0.05: 6 
padj <= 0.05: 3 
|log2FoldChange| >= 2: 0 

Warning message:
package ‘S4Vectors’ was built under R version 4.5.3 
estimating size factors
estimating dispersions
gene-wise dispersion estimates
mean-dispersion relationship
final dispersion estimates
fitting model and testing

[OK] results_1mo_4mo.psiclass.csv
[OK] results_1mo_4mo.pval05.psiclass.csv
[OK] results_1mo_4mo.qval05.psiclass.csv
[OK] normalized_counts.1mo_4mo.psiclass.csv


In [79]:
# Summarize DESeq2 results and identify GENCODE genes
# Read all columns as text to preserve PsiCLASS gene IDs
deseq_results = pd.read_csv(
    RESULTS_FILE,
    index_col=0,
    dtype=str)

# Convert statistical columns to numeric values
numeric_columns = [
    "baseMean",
    "log2FoldChange",
    "lfcSE",
    "stat",
    "pvalue",
    "padj"
]

for column in numeric_columns:
    deseq_results[column] = pd.to_numeric(
        deseq_results[column],
        errors="coerce")

# Select loci with p-value <= 0.05
deseq_significant = deseq_results.loc[
    deseq_results["pvalue"].le(0.05)
].copy()

# Map PsiCLASS loci to reference GENCODE genes
deseq_significant["GENCODE_gene"] = [
    ", ".join(sorted(gene_mapping.get(gene_id, set())))
    or "No reference annotation"
    for gene_id in deseq_significant.index]

# Display significant loci
display(
    deseq_significant[
        [
            "GENCODE_gene",
            "baseMean",
            "log2FoldChange",
            "pvalue",
            "padj"
        ]
    ].sort_values("pvalue"))

# Count distinct reference gene names
significant_gene_names = set()

for gene_id in deseq_significant.index:
    significant_gene_names.update(
        gene_mapping.get(gene_id, set()))

print("Loci with p-value <= 0.05:", len(deseq_significant))
print("Distinct GENCODE gene names:", len(significant_gene_names))
print("Genes:", ", ".join(sorted(significant_gene_names)))

# Verify Mag expression
mag_loci = [
    gene_id
    for gene_id in deseq_results.index
    if "Mag" in gene_mapping.get(gene_id, set())]

print("\nMag loci:")

for gene_id in mag_loci:
    print(
        f"{gene_id}: "
        f"baseMean = {deseq_results.loc[gene_id, 'baseMean']:.6f}")

,GENCODE_gene,baseMean,log2FoldChange,pvalue,padj
7.422,Mag,2101.912651,-1.144107,1.702659e-28,2.417775e-26
7.9,Sirt2,1393.567945,-0.904389,9.394424e-19,6.670041e-17
7.21,Sirt2,228.320688,-0.700047,7.132890e-06,3.376235e-04
7.395,Rps12-ps4,130.567504,0.552088,2.839258e-03,1.007936e-01
7.200,C230062I16Rik,198.621125,0.335331,2.664561e-02,7.567354e-01
7.335,Arhgap33,375.595972,-0.260698,3.209505e-02,7.595828e-01


Loci with p-value <= 0.05: 6
Distinct GENCODE gene names: 5
Genes: Arhgap33, C230062I16Rik, Mag, Rps12-ps4, Sirt2

Mag loci:
7.422: baseMean = 2101.912651


In [80]:
# Verify final DESeq2 results
# Summarize statistical significance
final_summary = pd.DataFrame({
    "Metric": [
        "Total loci",
        "Loci with p-value <= 0.05",
        "Loci with adjusted p-value <= 0.05",
        "Loci with |log2FoldChange| >= 2",
        "Distinct GENCODE genes (p-value <= 0.05)"
    ],
    "Count": [
        len(deseq_results),
        int(deseq_results["pvalue"].le(0.05).sum()),
        int(deseq_results["padj"].le(0.05).sum()),
        int(deseq_results["log2FoldChange"].abs().ge(2).sum()),
        len(significant_gene_names)
    ]})

display(final_summary)

# Verify Mag expression
mag_ids = [
    gene_id
    for gene_id in deseq_results.index
    if "Mag" in gene_mapping.get(gene_id, set())]

if not mag_ids:
    raise ValueError("Mag was not found in the annotation.")

for gene_id in mag_ids:
    print(
        f"Mag ({gene_id}) baseMean: "
        f"{deseq_results.loc[gene_id, 'baseMean']:.6f}"    )

,Metric,Count
0,Total loci,142
1,Loci with p-value <= 0.05,6
2,Loci with adjusted p-value <= 0.05,3
3,Loci with |log2FoldChange| >= 2,0
4,Distinct GENCODE genes (p-value <= 0.05),5


Mag (7.422) baseMean: 2101.912651


## 13. Final Project Questions

This section summarizes the results of the RNA-seq analysis workflow and provides the information needed to answer the final project questions.

Results are organized according to the four main analytical steps:

- **Q1–Q5:** STAR alignment statistics
- **Q6–Q10:** PsiCLASS transcript assembly
- **Q11–Q15:** Cuffdiff2 differential expression analysis
- **Q16–Q20:** DESeq2 differential expression analysis

All results are retrieved from the files generated in the previous sections. No additional alignment, transcript assembly, or differential expression analysis is performed.

Original PsiCLASS gene and transcript identifiers are preserved to ensure consistency between the annotation and differential expression results.

In [87]:
# STAR Alignment Statistics (Q1–Q5)
# Retrieve STAR alignment statistics for 1mo_Rep2

sample = "1mo_Rep2"

star_log = STAR_ALIGNMENT_DIR / sample / "Log.final.out"
star_bam = STAR_BAMS[sample]

if not star_log.is_file():
    raise FileNotFoundError(star_log)

# Read STAR alignment statistics
star_stats = {}

with star_log.open() as file:
    for line in file:
        if "|" in line:
            key, value = line.split("|", 1)
            star_stats[key.strip()] = value.strip()

# Count alignment records in the BAM file
bam_records = subprocess.run(
    ["samtools", "view", "-c", str(star_bam)],
    capture_output=True,
    text=True,
    check=True
)

# Summarize alignment statistics
alignment_summary = pd.DataFrame({
    "Metric": [
        "Number of input reads",
        "Uniquely mapped reads",
        "Reads mapped to multiple loci",
        "Reads unmapped: too short",
        "Total BAM alignment records"
    ],
    "Value": [
        int(star_stats["Number of input reads"]),
        int(star_stats["Uniquely mapped reads number"]),
        int(star_stats["Number of reads mapped to multiple loci"]),
        int(star_stats["Number of reads unmapped: too short"]),
        int(bam_records.stdout.strip())
    ]
})

display(alignment_summary)

,Metric,Value
0,Number of input reads,40653
1,Uniquely mapped reads,40076
2,Reads mapped to multiple loci,570
3,Reads unmapped: too short,1
4,Total BAM alignment records,41320


In [88]:
# PsiCLASS Transcript Assembly (Q6–Q10)
# Summarize PsiCLASS transcript assembly statistics
def summarize_gtf(gtf_path):
    gene_ids = set()
    transcript_ids = set()
    exon_counts = Counter()

    with gtf_path.open() as gtf:
        for line in gtf:
            if line.startswith("#") or not line.strip():
                continue

            fields = line.rstrip("\n").split("\t")

            if len(fields) != 9:
                continue

            attrs = dict(re.findall(r'(\w+)\s+"([^"]+)"', fields[8]))
            gene_id = attrs.get("gene_id")
            transcript_id = attrs.get("transcript_id")

            if gene_id:
                gene_ids.add(gene_id)
            if transcript_id:
                transcript_ids.add(transcript_id)
                if fields[2] == "exon":
                    exon_counts[transcript_id] += 1

    return {
        "Genes (loci)": len(gene_ids),
        "Transcripts": len(transcript_ids),
        "Single-exon transcripts": sum(
            count == 1 for count in exon_counts.values()
        ),
        "Multi-exon transcripts": sum(
            count > 1 for count in exon_counts.values())}

# Define consensus and sample-specific assemblies
assembly_files = {
    "Consensus": PSICLASS_VOTE,
    **{
        sample: PSICLASS_DIR / f"psiclass_sample_{i}.gtf"
        for i, sample in enumerate(SAMPLES)
    }
}

# Generate assembly summary
assembly_summary = pd.DataFrame.from_dict(
    {
        name: summarize_gtf(path)
        for name, path in assembly_files.items()
    },
    orient="index")

display(assembly_summary)

,Genes (loci),Transcripts,Single-exon transcripts,Multi-exon transcripts
Consensus,168,211,84,127
1mo_Rep1,173,204,85,119
1mo_Rep2,171,210,84,126
1mo_Rep3,173,212,86,126
4mo_Rep1,173,209,85,124
4mo_Rep2,172,211,84,127
4mo_Rep3,173,212,86,126


In [89]:
# Compare PsiCLASS transcript counting methods
sample_gtf = PSICLASS_DIR / "psiclass_sample_1.gtf"

if not sample_gtf.is_file():
    raise FileNotFoundError(sample_gtf)

# Count distinct transcript IDs from valid GTF records
transcript_count = assembly_summary.loc[
    "1mo_Rep2", "Transcripts"]

# Reproduce the original shell counting procedure
import shlex

command = (
    f"cut -f9 {shlex.quote(str(sample_gtf))} "
    "| cut -d ' ' -f4 | sort -u | wc -l")

result = subprocess.run(
    ["bash", "-c", command],
    capture_output=True,
    text=True,
    check=True)

# Count GTF comment lines
with sample_gtf.open() as gtf:
    comment_count = sum(
        line.startswith("#") for line in gtf)

print("Sample: 1mo_Rep2")
print("Distinct transcript IDs:", transcript_count)
print("Original shell command count:", int(result.stdout.strip()))
print("GTF comment lines:", comment_count)

Sample: 1mo_Rep2
Distinct transcript IDs: 210
Original shell command count: 213
GTF comment lines: 3


In [90]:
# Cuffdiff2 Differential Expression (Q11–Q15)
# Summarize Cuffdiff2 differential expression results
# Read gene-level differential expression results
cuff_results = pd.read_csv(GENE_DIFF, sep="\t", dtype={"gene_id": str})

# Convert numeric columns
for column in [
    "p_value",
    "q_value",
    "log2(fold_change)",
    "value_1",
    "value_2"
]:
    cuff_results[column] = pd.to_numeric(
        cuff_results[column],
        errors="coerce"    )

# Identify FDR-significant loci
fdr_significant = cuff_results[
    cuff_results["significant"]
    .astype(str)
    .str.lower()
    .eq("yes")
].copy()

# Identify loci meeting p-value and fold-change thresholds
pval_fc_significant = cuff_results[
    cuff_results["status"].eq("OK")
    & cuff_results["p_value"].le(0.05)
    & cuff_results["log2(fold_change)"].abs().ge(1)]

# Identify loci with FPKM >= 2 in either condition
expressed_loci = cuff_results[
    cuff_results["value_1"].ge(2)
    | cuff_results["value_2"].ge(2)]

# Identify distinct GENCODE genes
cuff_gene_names = set()

for gene_id in fdr_significant["gene_id"]:
    cuff_gene_names.update(
        gene_mapping.get(gene_id, set())    )

# Display summary
cuff_summary = pd.DataFrame({
    "Metric": [
        "Total loci",
        "FDR-significant loci",
        "Distinct GENCODE genes",
        "p-value <= 0.05 and |log2FC| >= 1",
        "FPKM >= 2 in either condition"
    ],
    "Count": [
        len(cuff_results),
        len(fdr_significant),
        len(cuff_gene_names),
        len(pval_fc_significant),
        len(expressed_loci)
    ]})

display(cuff_summary)

print("Significant GENCODE genes:")
print(", ".join(sorted(cuff_gene_names)))

# Display significant loci
display(
    fdr_significant[
        [
            "gene_id",
            "gene",
            "value_1",
            "value_2",
            "log2(fold_change)",
            "p_value",
            "q_value"
        ]
    ].sort_values("p_value"))

,Metric,Count
0,Total loci,142
1,FDR-significant loci,4
2,Distinct GENCODE genes,3
3,p-value <= 0.05 and |log2FC| >= 1,1
4,FPKM >= 2 in either condition,142


Significant GENCODE genes:
Mag, Rps12-ps4, Sirt2


,gene_id,gene,value_1,value_2,log2(fold_change),p_value,q_value
27,7.21,Sirt2,70387.2,42574.5,-0.725321,0.00005,0.003550
137,7.9,Sirt2,53308.3,29322.9,-0.862334,0.00005,0.003550
111,7.422,Mag,49035.6,23412.3,-1.066560,0.00070,0.033133
101,7.395,Rps12-ps4,20863.8,29808.9,0.514740,0.00105,0.037275


In [91]:
# Summarize DESeq2 results for the final project
# Read DESeq2 results while preserving PsiCLASS gene IDs
deseq_results = pd.read_csv(RESULTS_FILE, index_col=0, dtype=str)

# Convert statistical columns to numeric values
numeric_columns = [
    "baseMean",
    "log2FoldChange",
    "lfcSE",
    "stat",
    "pvalue",
    "padj"]

for column in numeric_columns:
    deseq_results[column] = pd.to_numeric(
        deseq_results[column],
        errors="coerce"    )

# Identify loci with p-value <= 0.05
significant_loci = deseq_results.loc[
    deseq_results["pvalue"].le(0.05)
].copy()

# Identify distinct GENCODE gene names
significant_genes = set()

for gene_id in significant_loci.index:
    significant_genes.update(
        gene_mapping.get(gene_id, set())    )

# Identify the Mag locus
mag_loci = [
    gene_id
    for gene_id in deseq_results.index
    if "Mag" in gene_mapping.get(gene_id, set())]

if len(mag_loci) != 1:
    raise ValueError(f"Expected one Mag locus, found {len(mag_loci)}."    )

mag_id = mag_loci[0]
mag_base_mean = deseq_results.loc[mag_id, "baseMean"]

# Summarize DESeq2 results
deseq_summary = pd.DataFrame({
    "Metric": [
        "Total loci",
        "Loci with p-value <= 0.05",
        "Distinct GENCODE genes (p-value <= 0.05)",
        "Loci with |log2FoldChange| >= 2",
        "Mag baseMean"
    ],
    "Value": [
        len(deseq_results),
        len(significant_loci),
        len(significant_genes),
        int(
            deseq_results["log2FoldChange"]
            .abs()
            .ge(2)
            .sum()
        ),
        round(float(mag_base_mean), 6)
    ]})

display(deseq_summary)

print("Significant GENCODE genes:")
print(", ".join(sorted(significant_genes)))
print(f"\nMag locus: {mag_id}")

,Metric,Value
0,Total loci,142.000000
1,Loci with p-value <= 0.05,6.000000
2,Distinct GENCODE genes (p-value <= 0.05),5.000000
3,Loci with |log2FoldChange| >= 2,0.000000
4,Mag baseMean,2101.912651


Significant GENCODE genes:
Arhgap33, C230062I16Rik, Mag, Rps12-ps4, Sirt2

Mag locus: 7.422


### Final Results Summary

- The RNA-seq workflow was completed using STAR, PsiCLASS, Cuffdiff2, and DESeq2.

**STAR alignment**

- For sample `1mo_Rep2`, STAR processed 40,653 input reads, including 40,076 uniquely mapped reads and 570 reads mapped to multiple loci.

**PsiCLASS transcript assembly**

- The consensus assembly contained 168 loci and 211 transcripts. The sample-specific assembly for `1mo_Rep2` contained 171 loci and 210 distinct transcript identifiers.

**Cuffdiff2 differential expression**

- Cuffdiff2 analyzed 142 loci and identified four FDR-significant loci associated with three distinct GENCODE gene names: *Mag*, *Rps12-ps4*, and *Sirt2*.

**DESeq2 differential expression**

- DESeq2 analyzed 142 loci across six biological samples. Six loci had unadjusted p-values ≤ 0.05, and three remained significant after FDR correction.
- The six loci with unadjusted p-values ≤ 0.05 were associated with five distinct GENCODE genes: *Arhgap33*, *C230062I16Rik*, *Mag*, *Rps12-ps4*, and *Sirt2*.
- The *Mag* locus (`7.422`) had a baseMean of approximately 2101.91.
- These results summarize the computational analyses performed in the final project.